# Phase 4 — Confidence, Date Logic, and Document Anomaly Validation

## Overview

Phase 4 focused on converting structured document extraction into a more trustworthy and reviewable decision layer.

By the end of Phase 3, the system could:

* extract structured fields from OCR,
* preserve OCR source-line references,
* validate whether extracted values were supported by OCR evidence,
* detect missing, invalid, or semantically weak evidence.

Phase 4 extended this by answering three additional questions:

1. **How reliable is each validated field?**
2. **Are the extracted dates logically consistent and operationally meaningful?**
3. **Does the document as a whole contain any important anomaly or risk?**

Phase 4 was divided into three parts:

* **Phase 4A — Field-Level Confidence**
* **Phase 4B — Date and Logical Validation**
* **Phase 4C — Document Anomaly Validation**

---

# Phase 4A — Field-Level Confidence

## Objective

The purpose of Phase 4A was to assign confidence to individual extracted fields without allowing the LLM to invent or estimate confidence scores.

The main design principle was:

> Field confidence must come from validated OCR evidence, not from the language model.

The LLM remained responsible only for structured extraction.

---

## Confidence Source

PaddleOCR already provides confidence values for each OCR text line.

For example, a licence field may depend on:

* a label line such as `LICENCE`,
* a value line containing the licence number.

The field confidence was therefore derived from the OCR lines referenced by the extracted field.

---

## Conservative Confidence Strategy

For fields supported by multiple OCR lines, the minimum OCR confidence among all supporting lines was used.

This conservative rule was selected because a field is only as trustworthy as its weakest required piece of evidence.

Example from the SIA badge:

* Licence label confidence: approximately 99.99%
* Licence number confidence: approximately 98.62%

Resulting field confidence:

**98.62%**

Similarly, the guard licence expiry date used both the `EXPIRES` label and the date value. Its field confidence became the minimum confidence across those supporting OCR lines.

---

## Evidence-Gated Confidence

Confidence was calculated only after Phase 3 evidence validation.

The behavior was defined as follows:

| Evidence State                  | Confidence Behavior             |
| ------------------------------- | ------------------------------- |
| Valid extracted field           | OCR-based confidence calculated |
| Field not extracted             | No confidence                   |
| Evidence validation failed      | No confidence                   |
| Supporting evidence unavailable | No trusted confidence           |

This prevented the system from assigning a high OCR score to a semantically unsupported field.

---

## Important Experiment — ID Card Date of Birth

The ID card OCR contained:

`23/08/2006`

The LLM interpreted this as:

`2006-08-23`

However, the OCR did not contain a reliable readable label such as:

* DOB
* DATE OF BIRTH
* BIRTH DATE

Phase 3 therefore returned:

**DATE_OF_BIRTH_CONTEXT_MISSING**

Phase 4A correctly refused to assign the OCR confidence of approximately 88.92% to the extracted date-of-birth field.

The field was instead classified as having invalid evidence.

### Key observation

A value may have a valid OCR confidence while still lacking sufficient semantic evidence.

Therefore:

> OCR confidence is not equivalent to field correctness.

---

## Real-Document Results

### SIA Badge

Validated fields included:

* full name,
* licence number,
* expiry date,
* issuer.

Representative confidence results were:

* Full name: approximately 96.43%
* Licence number: approximately 98.62%
* Expiry date: approximately 99.84%
* Issuer: approximately 99.98%

All validated fields received evidence-based confidence scores.

---

### Guard Licence

Validated fields included:

* full name,
* licence number,
* expiry date,
* date of birth,
* issue date,
* issuer.

Representative results included:

* Full name: approximately 99.99%
* Licence number: approximately 99.97%
* Expiry date: approximately 99.84%
* Date of birth: approximately 99.99%
* Issue date: approximately 99.90%
* Issuer: approximately 98.79%

---

### ID Card

Trusted fields included:

* full name,
* ID number.

The date of birth remained untrusted because its semantic context could not be verified.

This demonstrated that the confidence layer correctly depended on evidence validation rather than raw OCR probability alone.

---

## Phase 4A Conclusion

Phase 4A successfully established a deterministic field-level confidence mechanism based on OCR evidence.

The main conclusion was:

> Confidence should only be exposed after evidence has been validated.

This prevents high OCR confidence from creating false trust in semantically incorrect extractions.

---

# Phase 4B — Date and Logical Validation

## Objective

Phase 4B introduced deterministic date validation after evidence and confidence checks.

Its purpose was to verify:

* whether extracted dates were valid,
* whether they were logically possible,
* whether date relationships were internally consistent,
* whether a document was active, expired, or approaching expiry.

Only date fields with trusted evidence were processed.

---

## Date Fields Evaluated

The system evaluated:

* expiry date,
* date of birth,
* issue date.

Dates were expected to have already been normalized to:

`YYYY-MM-DD`

---

## Evidence Dependency

Phase 4B did not evaluate dates that had failed earlier evidence validation.

For example, the ID-card date of birth was extracted but had invalid semantic evidence.

Its logical-validation status was therefore:

**SKIPPED_INVALID_EVIDENCE**

This prevented the system from drawing downstream conclusions from uncertain upstream data.

---

## Expiry Status Logic

A configurable reference date was used for deterministic testing.

The following document states were supported:

* **ACTIVE**
* **EXPIRING_SOON**
* **EXPIRES_TODAY**
* **EXPIRED**
* **NOT_AVAILABLE**

An expiring-soon threshold of 30 days was used during the completed experiments.

---

## Logical Date Rules

The following logical checks were implemented and tested:

### Future Date of Birth

A date of birth cannot occur after the reference date.

Detection:

**FUTURE_DATE_OF_BIRTH**

---

### Future Issue Date

A document issue date cannot normally occur after the current reference date.

Detection:

**FUTURE_ISSUE_DATE**

---

### Expiry Before Issue Date

The expiry date should not occur before the document was issued.

Detection:

**EXPIRY_BEFORE_ISSUE_DATE**

---

### Date of Birth After Issue Date

A person's birth date cannot occur after the document issue date.

Detection:

**DOB_AFTER_ISSUE_DATE**

---

### Date of Birth After Expiry Date

A person's birth date cannot occur after the document expiry date.

Detection:

**DOB_AFTER_EXPIRY_DATE**

---

## Real-Document Tests

### Guard Licence

Validated dates:

* DOB: 1990-01-01
* Issue date: 2025-01-01
* Expiry date: 2026-01-01

Using the reference date 2026-08-14:

* Expiry status: **EXPIRED**
* Days until expiry: **-225**
* Logical issues: **None**

The chronological order was valid:

DOB → Issue Date → Expiry Date

The important conclusion was that an expired document is not necessarily logically invalid.

---

### SIA Badge

Validated expiry:

`2021-03-24`

Using the same reference date:

* Expiry status: **EXPIRED**
* Days until expiry: **-1969**
* No logical date inconsistencies were found.

The absence of DOB and issue date was handled without generating false errors.

---

### ID Card

The date of birth existed in the extraction but had previously failed semantic evidence validation.

Phase 4B correctly skipped it.

There was no validated expiry date.

Result:

* Expiry status: **NOT_AVAILABLE**
* DOB: **SKIPPED_INVALID_EVIDENCE**
* No unsupported logical conclusions were generated.

---

## Dedicated Negative Test Suite

A separate deterministic test suite was used to verify each date rule.

The following cases were successfully detected:

| Test                       | Result                   |
| -------------------------- | ------------------------ |
| Active document            | ACTIVE                   |
| Expiring within 30 days    | EXPIRING_SOON            |
| Expiring on reference date | EXPIRES_TODAY            |
| Already expired            | EXPIRED                  |
| Future DOB                 | FUTURE_DATE_OF_BIRTH     |
| Future issue date          | FUTURE_ISSUE_DATE        |
| Expiry before issue date   | EXPIRY_BEFORE_ISSUE_DATE |
| DOB after issue date       | DOB_AFTER_ISSUE_DATE     |
| DOB after expiry date      | DOB_AFTER_EXPIRY_DATE    |

All negative tests passed.

---

## Important Observation

Phase 4B deliberately separated:

* **evidence validity,**
* **logical validity,**
* **expiry status.**

For example, a document may simultaneously be:

* correctly extracted,
* supported by valid evidence,
* logically consistent,
* but expired.

This distinction is important for later human review and operational decisions.

---

## Phase 4B Conclusion

Phase 4B successfully introduced deterministic date reasoning without relying on the LLM.

The validator demonstrated correct handling of:

* valid dates,
* missing dates,
* invalid evidence,
* expiry states,
* impossible date relationships.

---

# Phase 4C — Document Anomaly Validation

## Objective

Phase 4C moved from individual-field validation to whole-document assessment.

Its purpose was to detect document-level conditions such as:

* missing critical information,
* weak trusted fields,
* conflicting identifiers,
* invalid evidence,
* date anomalies,
* expiry-related warnings,
* unknown document types.

This layer did not make the final human approval decision. It produced structured errors and warnings for later review.

---

## Document-Specific Critical Fields

Critical fields were defined according to the document type.

### SIA Badge

Critical fields:

* full name,
* licence number,
* expiry date,
* issuer.

### Guard Licence

Critical fields:

* full name,
* licence number,
* expiry date,
* issuer.

### ID Card

Critical fields:

* full name,
* ID number.

The ID-card date of birth was intentionally not made mandatory because the available sample demonstrated that OCR may capture the date while failing to capture reliable DOB context.

---

## Severity Model

Two main levels were used:

### ERROR

Represents a document condition that prevents the document from being considered fully valid.

Examples:

* missing critical field,
* invalid critical evidence,
* unknown document type,
* duplicate identifier mapping,
* logical date inconsistency.

### WARNING

Represents an important condition that does not necessarily invalidate the document structure.

Examples:

* expired document,
* document expiring soon,
* low-confidence critical field,
* non-critical field with invalid evidence.

---

## Anomaly Rules

### Missing Critical Field

If a document-specific required field was absent:

**MISSING_CRITICAL_FIELD**

This was treated as an error.

---

### Critical Field Not Trusted

If a critical field was present but had not passed evidence validation:

**CRITICAL_FIELD_NOT_TRUSTED**

This was treated as an error.

---

### Low Critical-Field Confidence

A 90% threshold was used during the completed experiments.

A trusted critical field below the threshold produced:

**LOW_CRITICAL_FIELD_CONFIDENCE**

This was treated as a warning rather than an automatic failure.

---

### Non-Critical Invalid Evidence

If an optional field existed but its evidence failed validation:

**EXTRACTED_FIELD_INVALID_EVIDENCE**

This was treated as a warning.

The ID-card DOB provided a real example of this behavior.

---

### Duplicate Identifier Mapping

If the same normalized identifier was assigned to both:

* licence number,
* ID number,

the system detected:

**DUPLICATE_IDENTIFIER_MAPPING**

This rule was particularly relevant because an earlier extraction experiment had demonstrated that numeric fields could be mapped to the wrong semantic field.

---

### Date Logical Issue Propagation

Logical errors produced in Phase 4B were propagated to the document anomaly layer.

For example:

**EXPIRY_BEFORE_ISSUE_DATE**

became a document-level error.

---

### Expired Document

A validated expired document generated:

**DOCUMENT_EXPIRED**

This was treated as a warning.

The document could therefore remain structurally valid while still being operationally expired.

---

### Expiring Soon

A document approaching its expiry threshold generated:

**DOCUMENT_EXPIRING_SOON**

This was also treated as a warning.

---

### Unknown Document Type

If document classification remained unknown:

**UNKNOWN_DOCUMENT_TYPE**

This was treated as an error.

---

# Prompt Refinements Identified During Phase 4

Phase 4 testing also exposed several upstream semantic extraction issues.

These were corrected through stricter extraction instructions.

## SIA Licence Number Mapping

At one point, the SIA licence number was mapped to `id_number` rather than `licence_number`.

The extraction rules were refined so that numbers associated with the `LICENCE` label on an SIA badge are mapped specifically to the licence-number field.

After the refinement, the expected SIA field mapping was consistently produced.

---

## Guard Licence Unlabelled Number

The guard licence contained an isolated OCR value:

`2301`

The LLM initially assigned this value to `id_number`, even though no ID label or semantic context existed.

The extraction rules were tightened so that:

> an isolated number must not automatically become an ID number.

After refinement:

* licence number remained correctly extracted,
* the unsupported `2301` value was ignored,
* ID number became null.

---

## Guard Licence Print Date

The OCR contained:

`PRINTDATE 01/01/2025`

The issue date was not always extracted consistently.

Both extraction and validation rules were updated to recognize `PRINTDATE` as valid issue-date context.

The final guard licence extraction correctly returned:

`2025-01-01`

as the issue date.

---

## Issuer Semantics

An early ID-card run incorrectly treated a corrupted national heading as the issuer.

The extraction rules were refined so that:

* country names,
* national slogans,
* republic headings,
* general document titles,

cannot automatically be treated as issuing authorities.

After refinement, the ID-card issuer correctly remained null when no specific authority could be verified.

---

# Real-Document Phase 4C Results

## SIA Badge

Result:

* Document valid: **Yes**
* Errors: **0**
* Warnings: **1**
* Warning: **DOCUMENT_EXPIRED**

All critical fields had trusted evidence.

---

## Guard Licence

After extraction refinements:

* full name: valid,
* licence number: valid,
* expiry date: valid,
* date of birth: valid,
* issue date: valid,
* issuer: valid,
* unsupported ID number: correctly omitted.

Result:

* Document valid: **Yes**
* Errors: **0**
* Warnings: **1**
* Warning: **DOCUMENT_EXPIRED**

---

## ID Card

Trusted core fields:

* full name,
* ID number.

DOB remained extracted but semantically unsupported.

Result:

* Document valid: **Yes**
* Errors: **0**
* Warnings: **1**
* Warning: **EXTRACTED_FIELD_INVALID_EVIDENCE**

This demonstrated that uncertain optional fields can be isolated without invalidating trusted core identity data.

---

# Dedicated Phase 4C Test Suite

A deterministic anomaly test suite was completed.

The following scenarios all passed:

| Test Case                     | Expected Result                  |
| ----------------------------- | -------------------------------- |
| Clean active document         | No anomalies                     |
| Missing critical field        | MISSING_CRITICAL_FIELD           |
| Low critical confidence       | LOW_CRITICAL_FIELD_CONFIDENCE    |
| Invalid critical evidence     | CRITICAL_FIELD_NOT_TRUSTED       |
| Duplicate identifier          | DUPLICATE_IDENTIFIER_MAPPING     |
| Date logical error            | Logical issue propagated         |
| Expired document              | DOCUMENT_EXPIRED                 |
| Expiring soon                 | DOCUMENT_EXPIRING_SOON           |
| Invalid non-critical evidence | EXTRACTED_FIELD_INVALID_EVIDENCE |
| Unknown document type         | UNKNOWN_DOCUMENT_TYPE            |

The final test suite completed successfully with all anomaly tests passing.

---

# Key Phase 4 Findings

Several important engineering findings emerged from Phase 4.

### 1. OCR confidence is not correctness

A high-confidence OCR value can still be assigned to the wrong semantic field.

### 2. Evidence validation must precede confidence

A field should not receive a trusted confidence score if its semantic evidence has failed.

### 3. Field presence does not prove field meaning

An OCR value such as a number or date may exist while its purpose remains unclear.

### 4. Null is safer than unsupported inference

When field meaning cannot be established reliably, returning null is preferable to guessing.

### 5. Expiry and validity are different concepts

A document can be structurally valid and logically consistent while still being expired.

### 6. Warnings and errors require different treatment

Operational conditions such as expiry should not necessarily be treated the same as missing or logically impossible information.

### 7. Downstream logic must respect upstream trust

Fields with invalid evidence should not be used for later date or anomaly reasoning.

### 8. Whole-document validation adds value beyond field validation

A document may contain individually valid fields while still having missing critical information, conflicting identifiers, or operational risks.

---

# Final Phase 4 Architecture

The completed validation pipeline is now:

```text
Document
   ↓
OCR Extraction
   ↓
Structured LLM Extraction
   ↓
Schema Validation
   ↓
Evidence Validation
   ↓
Phase 4A — Field-Level Confidence
   ↓
Phase 4B — Date & Logical Validation
   ↓
Phase 4C — Document Anomaly Validation
   ↓
Trusted / Warning / Error Document State
```

---

# Final Conclusion

Phase 4 successfully transformed the system from a structured extraction pipeline into a deterministic trust and validation layer.

The completed system can now:

* assign evidence-based confidence to trusted fields,
* refuse confidence for unsupported fields,
* evaluate expiry state,
* detect impossible date relationships,
* identify missing critical information,
* detect duplicate identifier mappings,
* distinguish errors from warnings,
* propagate earlier validation failures,
* and produce a document-level trust state.

All three real document types were tested, and dedicated negative test suites were completed for both date logic and document anomaly detection.

**Phase 4A — Field-Level Confidence: Complete**

**Phase 4B — Date and Logical Validation: Complete**

**Phase 4C — Document Anomaly Validation: Complete**

**Phase 4: Complete**
